# SGT runner — Kaggle (T4×2 / P100)

Run this for the **0.5B sweep** (primary path now that Colab idle-disconnect proved fragile) and the **1.5B sweep**. Internet must be ON in notebook settings; GPU accelerator must be set to T4 x2 or P100.

Outputs go to `/kaggle/working/runs/`; download the zip at session end (cell 4) or commit the notebook output.

**Before running:** confirm `experiments/sgt/preregistration.md` is committed in the repo head.


In [ ]:
%cd /kaggle/working
!rm -rf prism
!git clone --depth 1 https://github.com/humanaiconvention/prism
%cd /kaggle/working/prism
# Verify cuda toolkit version Kaggle is shipping; cu121 is the typical default in 2025-2026.
!nvidia-smi | head -5
!pip install -q -e . transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 bitsandbytes==0.44.1
# Pin torch to the locked version. cu121 matches typical Kaggle T4 runtime; flip to cu118 if nvidia-smi shows CUDA 11.x.
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
# Kaggle's base image preinstalls torchvision/torchaudio at the kernel's torch version; after we pin torch=2.5.1, the
# C++ ABI breaks (operator torchvision::nms not found) and `from transformers import AutoModelForCausalLM` blows up.
# SGT does not import torchvision/torchaudio, so the simplest fix is to remove them.
!pip uninstall -q -y torchvision torchaudio


In [ ]:
import os, sys, subprocess
sys.path.insert(0, '/kaggle/working/prism/src')
# Default to 0.5B sweep. Switch MODEL/TEACHER and RUNS_DIR for the 1.5B sweep.
MODEL = 'Qwen/Qwen2.5-0.5B'
TEACHER = 'Qwen/Qwen2.5-1.5B'   # for R4 only on 0.5B; for 1.5B switch to Qwen/Qwen2.5-7B (4-bit)
RUNS_DIR = '/kaggle/working/runs/0p5b'
os.makedirs(RUNS_DIR, exist_ok=True)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.device_count(), 'devices')
for i in range(torch.cuda.device_count()):
    print(f'  cuda:{i} = {torch.cuda.get_device_name(i)}')


In [ ]:
# Full 0.5B sweep — 5 regimes × 3 seeds = 15 runs, all in one cell so Kaggle treats it as active work.
# Per-run wallclock with batched make_synthetic (commit 580090b9): ~30-50 min on T4 (slower than L4).
# Entire sweep estimate: ~10 GPU-hr; fits in one Kaggle session (12-hr cap) if you stay attentive,
# else split into two sessions and comment out completed (regime, seed) pairs on resume.
import time, traceback
REGIMES = ['R1', 'R1_accum', 'R2', 'R3', 'R4']
SEEDS = [11, 23, 42]
BATCH = [(r, s) for r in REGIMES for s in SEEDS]
for regime, seed in BATCH:
    out_file = f'{RUNS_DIR}/{regime}_{seed}.json'
    if os.path.exists(out_file):
        print(f'== SKIP {regime} seed={seed} (already exists)')
        continue
    cmd = ['python', '-u', 'sgt_runner.py', '--regime', regime, '--seed', str(seed),
           '--model', MODEL, '--out', RUNS_DIR]
    if regime == 'R4':
        cmd += ['--teacher', TEACHER]
    print('==== START', regime, 'seed=', seed, time.strftime('%H:%M:%S'), '====', flush=True)
    print('>>>', ' '.join(cmd), flush=True)
    t0 = time.time()
    try:
        subprocess.run(cmd, cwd='/kaggle/working/prism/experiments/sgt', check=True)
    except subprocess.CalledProcessError as e:
        traceback.print_exc()
        print(f'!!!! {regime} seed={seed} FAILED with exit code {e.returncode}; continuing', flush=True)
        continue
    print(f'==== DONE  {regime} seed={seed}  wallclock={time.time()-t0:.0f}s ====', flush=True)
print('ALL_DONE')


In [ ]:
# Verify outputs and zip for download
!ls -la {RUNS_DIR}
!cd /kaggle/working && zip -r runs_0p5b.zip runs/
!ls -la /kaggle/working/runs_0p5b.zip
